# 💬 Ejercicios: OpenAI
# ChatGPT - suscripciones, planes, actividad y churn

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales de OpenAI.

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué plan de pago
### aporta más ingreso
### recurrente (MRR)?

MRR = usuarios activos x precio.
El motor del negocio SaaS.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/openai_chatgpt.csv'
)

act = df[df['activo']]

r = (
  act
  .groupby('plan')['precio_mes_usd']
  .agg(['sum', 'size'])
  .rename(columns={
    'sum': 'mrr',
    'size': 'usuarios'})
)

r['%'] = (
  r['mrr']
  / r['mrr'].sum() * 100
).round(1)

print('MRR por plan:')
print(
  r.sort_values('mrr',
    ascending=False)
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué plan sufre más
### churn y nos vacía
### la base de usuarios?

Churn rate por plan. Retener es
más barato que captar.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/openai_chatgpt.csv'
)

r = (
  df
  .groupby('plan')['churn']
  .agg(['mean', 'size'])
)

r['churn_%'] = (
  r['mean'] * 100
).round(1)

print('Churn por plan:')
print(
  r[['churn_%', 'size']]
  .sort_values('churn_%',
    ascending=False)
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Los que más usan
### ChatGPT se van menos?
### (actividad vs churn)

Mensajes/día en tramos y su churn.
La actividad como señal de fuga.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/openai_chatgpt.csv'
)

df['uso'] = pd.qcut(
  df['mensajes_dia'], 4,
  labels=['bajo', 'medio',
          'alto', 'intenso'],
)

r = (
  df
  .groupby('uso',
    observed=True)['churn']
  .mean()
  .mul(100)
  .round(1)
)

print('Churn % por nivel de uso:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué % de usuarios
### Free convierte a plan
### de pago? (tasa pago)

Reparto Free vs pago y qué peso
tiene cada país en el pago.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/openai_chatgpt.csv'
)

df['paga'] = df['plan'] != 'Free'

tasa = df['paga'].mean() * 100

r = (
  df
  .groupby('pais')['paga']
  .mean()
  .mul(100)
  .round(1)
  .sort_values(ascending=False)
)

print(f'Tasa pago global: {tasa:.1f}%')
print('\n% de pago por país:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuánto vale un
### usuario a lo largo de
### su vida? (LTV simple)

LTV = precio x antigüedad media.
El valor acumulado por plan.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/openai_chatgpt.csv'
)

pago = df[df['plan'] != 'Free']

r = (
  pago
  .groupby('plan')
  .agg(
    precio=('precio_mes_usd',
            'mean'),
    meses=('antiguedad_meses',
           'mean'),
  )
)

r['ltv_usd'] = (
  r['precio'] * r['meses']
).round(0)

print('LTV estimado por plan:')
print(
  r.round(1)
  .sort_values('ltv_usd',
    ascending=False)
)